# Faster R-CNN ResNet50-FPN experiment

This notebook runs the first pretrained CNN object detector while keeping the accepted grouped train/validation split and the ResNet18 regression control unchanged. It records the run locally in MLflow, including epoch curves, configuration, source revision, validation predictions, a best checkpoint, and a native MLflow PyTorch model.

The primary model-selection metric is mean IoU on the fixed validation split. For this one-box task, each image uses the highest-confidence class-1 detection after the detector's NMS. A missing or invalid detection receives IoU 0 and is also counted in prediction coverage. Predictions are mapped from the letterboxed detector canvas back to original-image coordinates before scoring.

Validation component and total losses are logged as diagnostics. Region-proposal sampling can be stochastic, and horizontal flips are applied only to training images, so the train and validation curves are not exact measurements of the same objective. Validation mean IoU remains the checkpoint-selection metric.

This is one seed-42 run. Repeat promising finalists with three predeclared seeds before concluding that an improvement is robust. Do not use the competition test set or leaderboard for model selection.

In [ ]:
from pathlib import Path
import sys

# Find the project root from either the repository folder or its parent.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / "data" / "raw" / "Train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    nested_project = Path.cwd() / "SEA_TURTLE_FACE_DETECTION"
    if (nested_project / "data" / "raw" / "Train.csv").is_file():
        PROJECT_ROOT = nested_project
    else:
        raise FileNotFoundError("Open this notebook from the project root or its parent directory.")

# Add the local package source so the notebook uses the current project code.
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT.resolve()}")

## Imports and fixed experiment configuration

TorchVision Faster R-CNN expects RGB float tensors in the range `[0, 1]`; it applies its own mean/std normalization. The shared data pipeline therefore disables its ImageNet normalization for this notebook. Letterboxing remains fixed at 384x512, and horizontal flip is used only for the training split.

In [ ]:
import hashlib
import platform

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
import torchvision
from IPython.display import display

from turtle_detection.data_pipeline import DEFAULT_TARGET_SIZE, create_data_loaders
from turtle_detection.detection import (
    build_faster_rcnn_resnet50_fpn,
    fit_faster_rcnn,
    seed_experiment,
)
from turtle_detection.mlflow_tracking import (
    append_experiment_summary,
    configure_local_mlflow,
    get_git_metadata,
    log_native_faster_rcnn_model,
)

SEED = 42
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 10
TARGET_SIZE = DEFAULT_TARGET_SIZE
TRAINABLE_BACKBONE_LAYERS = 3
LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
SCHEDULER_STEP_SIZE = 3
SCHEDULER_GAMMA = 0.1
BOX_DETECTIONS_PER_IMAGE = 100

# Fix Python, NumPy, and PyTorch RNG seeds before data shuffling and model creation.
seed_experiment(SEED)
# Use the available CUDA device, falling back to CPU for compatible runs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "mlflow": mlflow.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
})

## Load the accepted split

The split CSV is not regenerated here. Its SHA-256 is logged with the run so that later comparisons can verify they used the same assignments.

In [ ]:
# Hash the accepted assignments so MLflow can identify the exact split used.
SPLIT_PATH = PROJECT_ROOT / "data" / "splits" / "train_validation.csv"
SPLIT_SHA256 = hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest()

# Faster R-CNN applies ImageNet normalization internally, so keep dataset tensors in [0, 1].
train_loader, validation_loader = create_data_loaders(
    PROJECT_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    target_size=TARGET_SIZE,
    normalize=False,
)

# Fail early if the local split or image preprocessing no longer matches the audited setup.
assert len(train_loader.dataset) == 1062
assert len(validation_loader.dataset) == 264
images, targets = next(iter(train_loader))
assert tuple(images[0].shape) == (3, *TARGET_SIZE)
assert 0.0 <= float(images[0].min()) <= float(images[0].max()) <= 1.0
print({
    "train_images": len(train_loader.dataset),
    "validation_images": len(validation_loader.dataset),
    "split_sha256": SPLIT_SHA256,
    "batch_images": len(images),
    "tensor_shape": tuple(images[0].shape),
    "tensor_min": float(images[0].min()),
    "tensor_max": float(images[0].max()),
})

## Build the pretrained detector

The COCO-pretrained classification head is replaced with a two-class head: background and turtle face. The detector's score threshold is set to zero so the predeclared top-confidence rule, rather than an arbitrary confidence cutoff, selects the single box. TorchVision NMS remains active at IoU 0.5.

In [ ]:
# Build the pretrained detector with the agreed single-face class and post-NMS scoring policy.
model = build_faster_rcnn_resnet50_fpn(
    pretrained=True,
    trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS,
    min_size=TARGET_SIZE[0],
    max_size=TARGET_SIZE[1],
    box_score_thresh=0.0,
    box_nms_thresh=0.5,
    box_detections_per_img=BOX_DETECTIONS_PER_IMAGE,
)
model.to(DEVICE)
print(f"Trainable parameters: {sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad):,}")

## Register the experiment in local MLflow

MLflow stores run metadata in `outputs/mlflow/tracking.db` and artifacts under `outputs/mlflow/artifacts/`. Both locations are ignored by Git and remain on this computer. The model is logged in MLflow's native PyTorch flavor; no Zindi images are uploaded to a remote service.

In [ ]:
# Store run metadata and artifacts in the project-local MLflow tracking directory.
tracking = configure_local_mlflow(PROJECT_ROOT, "sea-turtle-face-detection")
git_metadata = get_git_metadata(PROJECT_ROOT)

split = pd.read_csv(SPLIT_PATH)
split_counts = split["split"].value_counts().to_dict()
assert split_counts == {"train": 1062, "validation": 264}
# Record split identity, preprocessing, optimizer, model, and runtime provenance with the run.
run_params = {
    "model_family": "object_detection",
    "architecture": "Faster R-CNN ResNet50-FPN",
    "pretrained_weights": "FasterRCNN_ResNet50_FPN_Weights.COCO_V1",
    "num_classes_including_background": 2,
    "split_id": "accepted_group_train_validation_split",
    "split_sha256": SPLIT_SHA256,
    "train_images": len(train_loader.dataset),
    "validation_images": len(validation_loader.dataset),
    "split_seed": 42,
    "experiment_seed": SEED,
    "target_size": f"{TARGET_SIZE[1]}x{TARGET_SIZE[0]}",
    "input_range": "float32_0_1",
    "preprocessing": "letterbox; detector applies ImageNet normalization internally",
    "augmentation": "horizontal_flip_train_only_p_0.5",
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "trainable_backbone_layers": TRAINABLE_BACKBONE_LAYERS,
    "optimizer": "SGD",
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "StepLR",
    "scheduler_step_size": SCHEDULER_STEP_SIZE,
    "scheduler_gamma": SCHEDULER_GAMMA,
    "epochs": EPOCHS,
    "box_score_threshold": 0.0,
    "box_nms_threshold": 0.5,
    "box_detections_per_image": BOX_DETECTIONS_PER_IMAGE,
    "resize_interpolation": "bilinear",
    "padding_color": "black",
    "single_box_policy": "highest_confidence_class_1_after_nms",
    "missing_detection_iou": 0.0,
    "selection_metric": "validation_mean_iou_original_image_coordinates",
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "mlflow_version": mlflow.__version__,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "cuda_runtime": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "source_commit": git_metadata["source_commit"],
    "source_dirty": git_metadata["source_dirty"],
}
print(tracking)
display(pd.Series(run_params, name="value").to_frame())

## Train and log curves, metrics, checkpoint, and native model

SGD uses a learning rate of 0.005, momentum of 0.9, and weight decay of 0.0005 to update the trainable detector weights. StepLR reduces the learning rate by a factor of 10 every 3 epochs. The best checkpoint is selected using mean IoU across the full validation split. This cell performs the full 10-epoch run; do not use a shortened smoke run for model comparison.

In [ ]:
# Train only parameters marked trainable by the detector builder.
optimizer = torch.optim.SGD(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=SCHEDULER_STEP_SIZE,
    gamma=SCHEDULER_GAMMA,
)

# Keep all metrics and artifacts for this training run grouped under one MLflow run.
with mlflow.start_run(run_name=f"fasterrcnn-r50fpn-seed-{SEED}") as active_run:
    mlflow.log_params(run_params)
    source_files = [
        PROJECT_ROOT / "notebooks" / "04_faster_rcnn_experiment.ipynb",
        PROJECT_ROOT / "src" / "turtle_detection" / "data_pipeline.py",
        PROJECT_ROOT / "src" / "turtle_detection" / "box_geometry.py",
        PROJECT_ROOT / "src" / "turtle_detection" / "detection.py",
        PROJECT_ROOT / "src" / "turtle_detection" / "mlflow_tracking.py",
        PROJECT_ROOT / "pyproject.toml",
        PROJECT_ROOT / "requirements.txt",
    ]
    for source_file in source_files:
        mlflow.log_artifact(str(source_file), artifact_path="source")
    mlflow.set_tags({
        "experiment_stage": "first_pretrained_cnn_detector",
        "evaluation_protocol": "single_top_confidence_box_on_fixed_validation_split",
        "data_location": "local_only",
    })
    checkpoint_path = (
        PROJECT_ROOT
        / "outputs"
        / "checkpoints"
        / f"fasterrcnn_resnet50_fpn_seed{SEED}_{active_run.info.run_id}_best.pt"
    )

    # Log per-epoch curves using epoch numbers as steps for aligned comparisons.
    def log_epoch(row):
        step = int(row["epoch"])
        mlflow.log_metric("optimizer/learning_rate", row["learning_rate"], step=step)
        for name, value in row.items():
            if name.startswith("train_"):
                metric_name = "train/" + name.removeprefix("train_")
            elif name.startswith("val_"):
                metric_name = "validation/" + name.removeprefix("val_")
            else:
                continue
            mlflow.log_metric(metric_name, value, step=step)

    # The helper validates every epoch and restores the checkpoint with best validation mean IoU.
    result = fit_faster_rcnn(
        model=model,
        train_loader=train_loader,
        validation_loader=validation_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        epochs=EPOCHS,
        checkpoint_path=checkpoint_path,
        target_size=TARGET_SIZE,
        on_epoch_end=log_epoch,
    )

    # Persist the best checkpoint, detailed predictions, and a loadable native PyTorch flavor.
    mlflow.log_metric("selection/best_epoch", result["best_epoch"])
    for name, value in result["validation_metrics"].items():
        mlflow.log_metric(f"best_validation/{name}", value)
    mlflow.log_artifact(str(checkpoint_path), artifact_path="checkpoints")
    mlflow.log_text(
        pd.DataFrame(result["history"]).to_csv(index=False),
        artifact_file="metrics/epoch_metrics.csv",
    )
    mlflow.log_text(
        pd.DataFrame(result["per_image_metrics"]).to_csv(index=False),
        artifact_file="validation/per_image_metrics.csv",
    )
    model_info = log_native_faster_rcnn_model(
        model=model,
        project_root=PROJECT_ROOT,
        target_size=TARGET_SIZE,
    )
    mlflow.set_tag("native_model_uri", model_info.model_uri)

    RUN_ID = active_run.info.run_id
    summary_row = {
        "experiment_id": f"fasterrcnn-r50fpn-seed{SEED}-{RUN_ID[:8]}",
        "run_date": pd.Timestamp.now().date().isoformat(),
        "git_commit": git_metadata["source_commit"],
        "git_dirty": git_metadata["source_dirty"],
        "mlflow_run_id": RUN_ID,
        "native_model_uri": model_info.model_uri,
        "split_sha256": SPLIT_SHA256,
        "model_family": "object_detection",
        "architecture": "Faster R-CNN ResNet50-FPN",
        "pretrained_weights": "FasterRCNN_ResNet50_FPN_Weights.COCO_V1",
        "head": "background_and_turtle_face",
        "split_id": "accepted_group_train_validation_split",
        "train_images": len(train_loader.dataset),
        "validation_images": len(validation_loader.dataset),
        "input_size": f"{TARGET_SIZE[1]}x{TARGET_SIZE[0]}",
        "preprocessing": "letterbox_float32_0_1_internal_imagenet_normalization",
        "augmentation": "horizontal_flip_train_only_p=0.5",
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "optimizer": "SGD+StepLR",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "epochs": EPOCHS,
        "best_epoch": result["best_epoch"],
        "selection_metric": "validation_mean_iou",
        "validation_mean_iou": result["validation_metrics"]["mean_iou"],
        "validation_median_iou": result["validation_metrics"]["median_iou"],
        "validation_mean_loss": next(
            row["val_loss_total"]
            for row in result["history"]
            if int(row["epoch"]) == result["best_epoch"]
        ),
        "validation_prediction_coverage": result["validation_metrics"]["prediction_coverage"],
        "validation_fraction_iou_ge_0_5": result["validation_metrics"]["fraction_iou_ge_0_5"],
        "validation_fraction_iou_ge_0_75": result["validation_metrics"]["fraction_iou_ge_0_75"],
        "validation_mean_selected_confidence": result["validation_metrics"]["mean_selected_confidence"],
        "validation_mean_class_one_detections": result["validation_metrics"]["mean_class_one_detections"],
        "checkpoint_path": checkpoint_path.relative_to(PROJECT_ROOT).as_posix(),
        "notes": "Single seed; best epoch selected by fixed-split mean IoU; detailed metrics and artifacts are in MLflow.",
    }
    experiments_csv = append_experiment_summary(PROJECT_ROOT, summary_row)

print({
    "run_id": RUN_ID,
    "best_epoch": result["best_epoch"],
    "best_checkpoint": str(result["checkpoint_path"]),
    "native_model_uri": model_info.model_uri,
    "experiment_registry": str(experiments_csv),
})
display(pd.DataFrame([summary_row]))
display(pd.Series(result["validation_metrics"], name="validation_value").to_frame())

## Inspect validation curves

In [ ]:
# Show localization quality and compare train/validation loss when both were logged.
history = pd.DataFrame(result["history"])
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(history["epoch"], history["val_mean_iou"], marker="o")
axes[0].axvline(result["best_epoch"], color="black", linestyle="--", label="best epoch")
axes[0].set(title="Validation mean IoU", xlabel="Epoch", ylabel="IoU", ylim=(0, 1))
axes[0].legend()

if "train_loss_total" in history:
    axes[1].plot(history["epoch"], history["train_loss_total"], marker="o", label="training")
if "val_loss_total" in history:
    axes[1].plot(history["epoch"], history["val_loss_total"], marker="o", label="validation")
else:
    axes[1].text(
        0.5, 0.5,
        "Validation loss unavailable in this history;\nshowing training loss only.",
        ha="center", va="center", transform=axes[1].transAxes,
    )
axes[1].set(title="Total loss", xlabel="Epoch", ylabel="Loss")
if axes[1].lines:
    axes[1].legend()

component_columns = sorted(
    column for column in history.columns
    if column.startswith("val_loss_") and column != "val_loss_total"
)
for column in component_columns:
    axes[2].plot(history["epoch"], history[column], marker="o", label=column.removeprefix("val_loss_"))
axes[2].set(title="Validation loss components", xlabel="Epoch", ylabel="Loss")
if component_columns:
    axes[2].legend()
else:
    axes[2].text(
        0.5, 0.5, "Validation loss components unavailable in this history.",
        ha="center", va="center", transform=axes[2].transAxes,
    )
plt.tight_layout()

## Review qualitative validation cases

The gallery draws the ground-truth box in green and the selected prediction in red on original images. Inspect misses and low-IoU cases before accepting the detector. These local images are displayed in the notebook and are not logged as MLflow artifacts.

In [ ]:
from PIL import Image, ImageDraw

# Review the weakest IoU cases alongside a reproducible random sample.
per_image = pd.DataFrame(result["per_image_metrics"]).sort_values("iou")
review_rows = pd.concat([per_image.head(3), per_image.sample(min(3, len(per_image)), random_state=SEED)])
review_rows = review_rows.drop_duplicates("image_id").head(6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis, (_, row) in zip(axes.flat, review_rows.iterrows()):
    dataset = validation_loader.dataset
    image_path = dataset._image_path(row["image_id"])
    with Image.open(image_path) as opened:
        image = opened.convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    # Convert normalized original-image coordinates to pixels for the overlay.
    gt_box = (
        row["gt_x"] * width,
        row["gt_y"] * height,
        (row["gt_x"] + row["gt_w"]) * width,
        (row["gt_y"] + row["gt_h"]) * height,
    )
    pred_box = (
        row["pred_x"] * width,
        row["pred_y"] * height,
        (row["pred_x"] + row["pred_w"]) * width,
        (row["pred_y"] + row["pred_h"]) * height,
    )
    draw.rectangle(gt_box, outline="lime", width=3)
    if row["has_valid_prediction"]:
        draw.rectangle(pred_box, outline="red", width=3)
    axis.imshow(image)
    axis.set_title(f"{row['image_id']} | IoU={row['iou']:.3f}")
    axis.axis("off")
for axis in axes.flat[len(review_rows):]:
    axis.axis("off")
plt.tight_layout()

## Open the local MLflow UI

From the project root, start the server in a separate PowerShell terminal:

```powershell
.\.venv\Scripts\mlflow.exe server --backend-store-uri sqlite:///outputs/mlflow/tracking.db --host 127.0.0.1 --port 5000
```

Then open `http://127.0.0.1:5000`. Each MLflow run contains parameters, step-by-step metrics, per-image validation results, the best checkpoint, and the native PyTorch model. Keep the database and artifacts local; they include experiment results derived from competition data.